# သင်ခန်းစာ 09 - ကိုယ်တိုင်၏ စဉ်းစားပုံအပေါ် စဉ်းစားခြင်း ဒီဇိုင်းပုံစံ


## တပ်ဆင်မှု

ဒီ notebook သည် Microsoft Agent Framework ကို အသုံးပြု၍ Metacognition ဒီဇိုင်းပုံစံကို ပြသသည်။

**လိုအပ်ချက်များ:**
- Azure OpenAI deployment ကို environment variables ဖြင့် ဖွဲ့စည်းထားရပါမည်
- Azure CLI အတွက် အတည်ပြုထားရပါမည် (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Metacognition ဆိုတာဘာလဲ?

Metacognition ဆိုသည်မှာ **စဉ်းစားခြင်းအပေါ် စဉ်းစားခြင်း** ဖြစ်သည်။ AI agents ၏ ပတ်ဝန်းကျင်တွင်၊ ၎င်းသည် အောက်ပါအရာများ ပြုလုပ်နိုင်သည့် အေးဂျင့်များကို တည်ဆောက်ခြင်းကို ဆိုလိုသည်။

- **ကိုယ်တိုင် ပြန်လည်သုံးသပ်နိုင်ခြင်း** — ၎င်းတို့၏ ကိုယ်ပိုင် ထုတ်လွှင့်ချက်များနှင့် အတွေးအခေါ် လုပ်ငန်းစဉ်များအပေါ် ကိုယ်တိုင် စောင့်ကြည့် သုံးသပ်နိုင်ခြင်း
- **အမှားများကို တွေ့ရှိနိုင်ခြင်း** — တိတ်တိတ်ပျက်ကွက်သွားခြင်းမဖြစ်စေရန် သက်သာစွာ ပြန်လည်ဖြေရှင်းနိုင်ခြင်း
- **တုံ့ပြန်ချက်များကို အကဲဖြတ်နိုင်ခြင်း** — ၎င်းတို့၏ တုံ့ပြန်ချက်များ ပြည့်စုံပြီး အကူအညီဖြစ်စေမည်ဟုတ် မဟုတ်ကို သတ်မှတ်နိုင်ခြင်း
- **နည်းဗျူဟာကို ပြင်ဆင်လိုက်ကာ သင့်လျော်စေရန် လိုက်လျောညီထွေ ဖြစ်နိုင်ခြင်း** — စတင်နည်းလမ်း မအောင်မြင်ပါက မဟာဗျူဟာ ပြောင်းလဲ၍ အခြားနည်းလမ်းများကို အသုံးပြုနိုင်ခြင်း (ဥပမာ၊ backup စနစ်သို့ ပြန်သွားခြင်း)

Metacognitive အေးဂျင့်သည် မေးခွန်းများကိုသာ ဖြေဆိုခြင်းမဟုတ်ဘဲ၊ ၎င်း၏ ကိုယ်ပိုင် စွမ်းဆောင်ရည်ကို စောင့်ကြည့်ကာ ချက်ချင်း သင့်တော်သလို ပြင်ဆင်တတ်သည်။


## အဓိကနှင့် အထောက်အပံ့ ကိရိယာများ

ကိုယ်ပိုင်သိမြင်မှုတွင် တွေ့ရသော ပုံစံတစ်ခုမှာ **အစားထိုးဗျူဟာ** ဖြစ်သည်။

အေးဂျင့်သည် အရင်ဆုံး အဓိက ကိရိယာကို စမ်းကြည့်သည်; ၎င်း မအောင်မြင်ပါက (ဥပမာ၊ 404 အမှား) အေးဂျင့်သည် ထိုမအောင်မြင်မှုကို သိရှိကာ ရှင်းလင်းစွာ အထောက်အပံ့ ကိရိယာသို့ ပြောင်းလဲသွားသည်။

ဒါသည် အဓိက ဝန်ဆောင်မှုများကို မရနိုင်နိုင်သလို ဖြစ်နိုင်သော အမှန်လက်တွေ့ စနစ်များနှင့် ဆင်တူပြီး၊ အေးဂ်င့်များသည် အခြားလမ်းကြောင်းရွေးချယ်ရန် မပြောင်းမီ ကိုယ်တိုင် ပြဿနာကို သံသယမဲ့စွာ စစ်ဆေးရပါမည်။

အောက်တွင် လေကြောင်း ရှာဖွေရေး ကိရိယာ နှစ်ခုကို သတ်မှတ်ထားသည်။

- **အဓိက** — ပဲရစ်၊ တိုကျို နှင့် ဘာစီလိုနာကို ဖုံးလွှမ်းသည်
- **အထောက်အပံ့** — ဘာလင်၊ စစ်ဒနီး နှင့် နယူးယောက်မြို့ကို ဖုံးလွှမ်းသည်


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## အမှားပြန်လည်ကာကွယ်နိုင်ပြီး ကိုယ်တိုင်သုံးသပ်နိုင်သော အေးဂျင့်

အောက်ပါ အေးဂျင့်ကို ပင်မ ပျံသန်းစနစ်ကို ပထမဦးစွာ စမ်းသပ်ရန်၊ ပျက်ကွက်မှုများကို အသိအမှတ်ပြုရန် နှင့် ရိုးရှင်းစွာ နောက်ကာကွယ်စနစ်သို့ ပြန်ရွေ့ရန် ညွှန်ကြားထားသည်။ တစ်ခုချင်းစီ တုံ့ပြန်ချက်အပြီး၌ လျင်မြန်စွာ ကိုယ်တိုင် သုံးသပ်ကာ အသုံးပြုသူ၏ မေးခွန်းကို အပြည့်အဝ ဖြေခဲ့ပါသလား ဆိုတာကို စစ်ဆေးသည်။


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## ကိုယ်တိုင်အကဲဖြတ်မှု ပုံစံ

metacognition ၏ အခြားတစ်ဖက်ကတော့ **ကိုယ်တိုင်အကဲဖြတ်ခြင်း** ဖြစ်သည်: သီးခြားအေးဂျင့် (သို့မဟုတ် တစ်ကြိမ် ထပ်မံစစ်ဆေးသောအချိန်တွင် တူညီသော အေးဂျင့်) သည် တုံ့ပြန်ချက်ကို ပြည့်စုံမှု၊ မှန်ကန်မှုနှင့် အထောက်အကူပြုမှု အရ သုံးသပ်သည်။

  
အောက်တွင် ကျွန်ုပ်တို့သည် `ResponseEvaluator` အေးဂျင့်ကို ဖန်တီးပြီး ခရီးသွားအေးဂျင့်၏ တုံ့ပြန်ချက်များကို သုံးခုအတိုင်းအတာပေါ် မူတည်၍ အမှတ်ပေးမည်။


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## အနှစ်ချုပ်

ဤသင်ခန်းစာတွင် သင်သည် Microsoft Agent Framework ကို အသုံးပြု၍ **ကိုယ်တိုင်ပြန်လည်သုံးသပ်နိုင်သော အေးဂျင့်များ** ကို မည်သို့ တည်ဆောက်ရမည်ကို လေ့လာခဲ့သည်။

- **ကိုယ်တိုင်ပြန်လည်သုံးသပ်ခြင်း**: အေးဂျင့်များသည် မိမိတို့၏ အတွေးအကြောင်းများကို စောင့်ကြည့်ကာ ဖြစ်ပျက်ခဲ့သည့် အရာများကို ထင်ရှားရှင်းလင်းစွာ ဆက်သွယ်ပြောကြားသည်။
- **Fallbacks ဖြင့် အမှားပြန်လည်ကယ်တင်ခြင်း**: အဓိက + အစားထိုး ကိရိယာ ပုံစံဖြစ်ပြီး အေးဂျင့်သည် မလုပ်ဆောင်နိုင်မှုများ (ဥပမာ၊ 404 အမှားများ) ကို တွေ့ရှိလျှင် အလိုအလျောက် အခြားရင်းမြစ်ကို စမ်းသပ်ကြည့်သည်။
- **ကိုယ်တိုင်အကဲဖြတ်ခြင်း**: တုန့်ပြန်ချက်များကို ပြည့်စုံမှု၊ တိကျမှု နှင့် အကူအညီဖြစ်စေမှု အရ အမှတ်ပေးသတ်မှတ်နိုင်သည့် သီးခြား အကဲဖြတ် အေးဂျင့်တစ်ခု ရှိသည်။

ဤနမူနာများက အေးဂျင့်များကို ပိုမိုခိုင်မာ၊ ပိုမိုထင်ရှားရှင်းလင်းပြီး ယုံကြည်စိတ်ချရစေပြီး — ထုတ်လုပ်မှုတွင် တပ်ဆင်ရာတွင် အရေးကြီးသော အရည်အသွေးများဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
သတိပေးချက်:
ဤစာရွက်ကို AI ဘာသာပြန်ဝန်ဆောင်မှုဖြစ်သည့် [Co-op Translator](https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်စေရန် ကြိုးစားသော်လည်း အလိုအလျောက် ဘာသာပြန်ချက်များတွင် အမှားများ သို့မဟုတ် မမှန်ကန်မှုများ ပါဝင်နိုင်ကြောင်း သတိပြုပါ။ မူလစာရွက်ကို မူလဘာသာဖြင့် ရှိသည့် အရင်းအမြစ်ကို တရားဝင် အကျင့်အကျယ် ရင်းမြစ်အဖြစ် ယူဆသင့်သည်။ အရေးပါသော အချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသား ဘာသာပြန်သူ၏ ဘာသာပြန်ချက်ကို အသုံးပြုရန် အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းကြောင့် ဖြစ်ပေါ်လာနိုင်သည့် နားမလည်မှုများ သို့မဟုတ် အဓိပ္ပာယ်မှားဖတ်ခြင်းများအတွက် ကျွန်ုပ်တို့ တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
